> **INSTRUCTOR SOLUTIONS** — do not share with learners before the session.

# Part 9 · Notebook 04 — A dynamic hedge ratio with the Kalman filter

**Sessions:** S4 (Dynamic hedge ratio with the Kalman filter) · [Lesson plan](../../docs/lessons/PART_09_STATISTICAL_TRADING.md) · graded labs in [`labs/part09/`](../../labs/part09/)

**You will:**
1. See a rolling-OLS hedge ratio lag (or jitter around) a drifting one.
2. Write the Kalman filter for a regression whose coefficients move.
3. Compare both with the true β.
4. See a filter that adapts too fast swallow the spread it should trade.

How these notebooks work: the setup, data and plotting code is written for you. Cells marked **✍️ Your turn** need a few lines from you.
If your answer does not match yet, the notebook continues with the reference answer so nothing else breaks.
All data is synthetic with known parameters (true half-lives, true pairs, true hedge ratios, planted premia), so every estimator can be checked against the truth.

In [ ]:
import sys
from pathlib import Path
for d in (Path.cwd(), Path.cwd().parent):       # p9lib.py is in notebooks/part09/
    sys.path.insert(0, str(d))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import p9lib as p

p.use_course_style()

## 1. Hedge ratios drift

Business mix, index membership and leverage change, so the hedge ratio does too. The simplest adaptive estimate is a **rolling OLS** β over the last `window` bars (including today): `cov(y, x) / var(x)` over the window, which is exactly the OLS slope with an intercept. Pandas has `rolling(window).cov(other)` and `rolling(window).var()`.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def rolling_ols_beta(y, x, window):
    y, x = pd.Series(np.asarray(y, float)), pd.Series(np.asarray(x, float))
    return (y.rolling(window).cov(x) / x.rolling(window).var()).to_numpy()

d = p.drifting_beta_pair()                                # true β moves from 1.0 to 2.0 over 2,000 days
cases = [(d.y, d.x, 250), (d.y, d.x, 60)]
mine = [p.attempt(rolling_ols_beta, *c) for c in cases]
mine = p.check("rolling_ols_beta", mine, [p.rolling_ols_beta(*c) for c in cases])
fig, ax = plt.subplots(figsize=(10, 3.4))
ax.plot(d.index, d.beta, color="black", lw=2, label="true β")
ax.plot(d.index, mine[0], lw=1, label="rolling OLS, 250 days"); ax.plot(d.index, mine[1], lw=0.8, label="rolling OLS, 60 days")
ax.set_ylim(-1, 4); ax.legend(); ax.set_title("rolling OLS: long wrong swings, or noise"); plt.show()

Neither window works. Within a window `x` wanders only a little, so the slope is poorly identified: a stretch where `x` and the spread noise happen to move together sends the 250-day estimate far off for months. The 60-day window reacts faster but jumps around from week to week.

## 2. The Kalman filter

Treat `(α_t, β_t)` as a hidden state that follows a random walk, and `y_t = α_t + β_t·x_t + e_t` as a noisy observation of it. Each bar, with `F = [1, x_t]`:

1. **Predict:** `P = P + Vw` (the state may have moved; `Vw = δ/(1 − δ)·I`).
2. **Forecast error**, *before* the update: `e_t = y_t − F·θ`. Its variance is `q_t = F P Fᵀ + r`.
3. **Gain:** `K = P Fᵀ / q_t`.
4. **Update:** `θ = θ + K e_t`, `P = P − K F P`.

`e_t` is the spread as seen with yesterday's hedge, and `e_t / √q_t` is a z-score that scales itself.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def kalman_hedge(y, x, delta=1e-4, r=1e-3):
    y, x = np.asarray(y, float), np.asarray(x, float)
    n = y.size
    theta, P = np.zeros(2), np.eye(2)                     # the state (α, β) and its covariance
    Vw = delta / (1 - delta) * np.eye(2)                  # how far (α, β) may move in one bar
    beta, alpha, e, q = (np.empty(n) for _ in range(4))
    for t in range(n):
        F = np.array([1.0, x[t]])
        P = P + Vw                                        # predict: the state may have moved
        q[t] = F @ P @ F + r
        e[t] = y[t] - F @ theta
        K = P @ F / q[t]
        theta = theta + K * e[t]                          # update
        P = P - np.outer(K, F) @ P
        alpha[t], beta[t] = theta
    return beta, alpha, e, q

cases = [(d.y, d.x), (d.y, d.x, 1e-6, 0.25)]
mine = [p.attempt(kalman_hedge, *c) for c in cases]
mine = p.check("kalman_hedge", mine, [p.kalman_hedge(*c) for c in cases])
kb = mine[0][0]
print(f"bar 200: true β {d.beta.iloc[200]:.3f}, Kalman {kb[200]:.3f};  last bar: true {d.beta.iloc[-1]:.3f}, Kalman {kb[-1]:.3f}")

In [ ]:
ests = {"Kalman (δ 1e-4)": kb, "rolling OLS 60": p.rolling_ols_beta(d.y, d.x, 60),
        "rolling OLS 250": p.rolling_ols_beta(d.y, d.x, 250), "rolling OLS 500": p.rolling_ols_beta(d.y, d.x, 500)}
err = pd.Series({k: np.nanmean(np.abs(v - d.beta.to_numpy())[500:]) for k, v in ests.items()}, name="mean |β error| after bar 500")
display(err.round(3).to_frame())
fig, ax = plt.subplots(figsize=(10, 3.4))
ax.plot(d.index, d.beta, color="black", lw=2, label="true β")
ax.plot(d.index, kb, lw=1, label="Kalman"); ax.plot(d.index, ests["rolling OLS 250"], lw=1, label="rolling OLS 250")
ax.set_ylim(0, 3); ax.legend(); ax.set_title("the filter tracks the drifting hedge ratio"); plt.show()

The filter stays within a few hundredths of the true β all the way from 1 to 2; rolling OLS is about ten times further off, whatever the window.

## 3. Too fast, and the hedge eats the signal

`δ` sets how fast β may move. On a pair whose true β is **constant** (1.5), trade the filter's own z-score `e/√q` with `pairs_positions`, hedging each day with **yesterday's** β, over the same out-of-sample period as notebook 03. The observation noise `r` is the variance of the formation spread.

In [ ]:
pair = p.cointegrated_pair()
F = 750
r = float(p.engle_granger(pair.y.iloc[:F], pair.x.iloc[:F])["spread"].var())
rows = {}
for delta in (1e-8, 1e-7, 1e-6, 1e-5, 1e-4):
    b, _, e, q = p.kalman_hedge(pair.y, pair.x, delta, r)
    zk = e / np.sqrt(q)
    pos = p.pairs_positions(zk)
    pl = p.pair_pnl(pos, pair.y, pair.x, np.r_[b[0], b[:-1]])[F:]          # yesterday's hedge: no look-ahead
    tr = pos[F - 1:]
    rows[f"δ = {delta:g}"] = {"std of z": np.std(zk[F:]), "max |z|": np.abs(zk[F:]).max(),
                              "trades": int(((tr[1:] != 0) & (tr[1:] != tr[:-1])).sum()), "OOS Sharpe": p.sharpe(pl)}
pd.DataFrame(rows).T.round(2)

With a tiny `δ` the hedge barely moves and `e/√q` is close to the spread divided by its formation volatility: few trades, but good ones. As `δ` grows, β chases every wiggle of `y`, the forecast error shrinks and so does the z-score: from `δ = 1e-6`, `|z|` never reaches 2 and nothing trades (NaN Sharpe). The filter has explained the spread away. A drifting relationship needs a larger `δ` than a stable one; choose it by walk-forward (homework), never on the test period.

## Wrap-up

* Rolling OLS trades lag for noise; the Kalman filter adapts with far less of both when β really drifts.
* Use the forecast error with yesterday's hedge, and lag the hedge in the P&L.
* Too large a `δ` absorbs the spread (lesson plan, common mistake 8).
* Graded version: `labs/part09/week31_pairs` (`kalman_hedge`, `rolling_ols_beta`); Clinic W1 compares fixed, rolling and Kalman hedges out of sample.